# StockInsight – A‑share Data Analysis Workflow  
**Supporting Notebook for Track 4 Interactive Tool**  
*Data source: baostock (free, no registration)*

## 1. Problem Definition & Target User

**Analytical problem:**  
Retail investors and finance students need a quick, code‑based way to assess the risk‑return profile of individual A‑share stocks (China market). They want to see not only price trends but also volatility, drawdown, Sharpe ratio, and abnormal volume days.

**Target user / audience:**  
- Beginner‑level equity researchers  
- University students learning Python for finance  
- Self‑directed retail investors without access to Bloomberg/Wind  

**Business relevance:**  
Understanding single‑stock risk metrics supports better investment decisions and portfolio construction. The analysis directly feeds into a Streamlit dashboard (`app.py`) that makes these insights interactive.

## 2. Data Source & Acquisition

- **Source:** baostock (http://baostock.com) – free, no API key, stable in China  
- **Access date:** (fill in the actual date you run this notebook, e.g., 2026-04-21)  
- **Stock universe:** All A‑shares (Shanghai `6xxxxx`, Shenzhen `0xxxxx` or `3xxxxx`)  
- **Example code:** `000001` (Ping An Bank)  
- **Time range:** last 365 days from today (adjustable)  
- **Fields retrieved:** `date`, `open`, `high`, `low`, `close`, `volume` (forward adjusted, `adjustflag=2`)  

**Why baostock?**  
It provides clean, adjusted historical data without registration. The forward‑adjusted prices ensure returns are calculated correctly.

In [2]:
# Import required libraries
import baostock as bs
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from datetime import datetime, timedelta

print("Libraries imported successfully.")

Libraries imported successfully.


## 3. Data Loading Function (baostock)

Define a function to log in, fetch data, and return a cleaned DataFrame.  
No hard‑coded paths – all parameters are passed as arguments.

In [3]:
def fetch_a_stock_data(symbol, start_date, end_date):
    """
    Fetch A-share historical daily data (forward adjusted) using baostock.
    
    Parameters:
        symbol (str): 6-digit stock code, e.g., '000001'
        start_date (str): 'YYYY-MM-DD'
        end_date (str): 'YYYY-MM-DD'
    
    Returns:
        df (pd.DataFrame): indexed by Date, columns Open, High, Low, Close, Volume
        stock_name (str): symbol as name (baostock does not provide name)
    """
    try:
        # Login to baostock
        lg = bs.login()
        if lg.error_code != '0':
            print(f"baostock login failed: {lg.error_msg}")
            return None, None
        
        # Format code: sh.xxxxxx for Shanghai, sz.xxxxxx for Shenzhen
        if symbol.startswith('6'):
            code = f"sh.{symbol}"
        else:
            code = f"sz.{symbol}"
        
        # Query history (forward adjusted)
        rs = bs.query_history_k_data_plus(
            code=code,
            fields="date,open,high,low,close,volume",
            start_date=start_date,
            end_date=end_date,
            frequency="d",
            adjustflag="2"   # forward adjusted
        )
        
        # Collect data rows
        data_list = []
        while rs.next():
            data_list.append(rs.get_row_data())
        
        bs.logout()
        
        if not data_list:
            return None, None
        
        # Convert to DataFrame
        df = pd.DataFrame(data_list, columns=['Date', 'Open', 'High', 'Low', 'Close', 'Volume'])
        for col in ['Open', 'High', 'Low', 'Close', 'Volume']:
            df[col] = pd.to_numeric(df[col], errors='coerce')
        df['Date'] = pd.to_datetime(df['Date'])
        df.set_index('Date', inplace=True)
        
        stock_name = symbol  # placeholder
        return df, stock_name
        
    except Exception as e:
        print(f"Error in fetch_a_stock_data: {e}")
        return None, None

## 4. Data Loading (Example)

Load data for stock `000001` over the last 365 days.  
All dates are computed relative to today – no hard‑coded paths.

In [4]:
# Set parameters
symbol = "000001"
end_date = datetime.today()
start_date = end_date - timedelta(days=365)

start_str = start_date.strftime("%Y-%m-%d")
end_str = end_date.strftime("%Y-%m-%d")

print(f"Fetching {symbol} from {start_str} to {end_str}...")
data, stock_name = fetch_a_stock_data(symbol, start_str, end_str)

if data is not None:
    print(f"Data shape: {data.shape}")
    print(data.head())
else:
    print("Failed to fetch data.")

Fetching 000001 from 2025-04-21 to 2026-04-21...
login success!
logout success!
Data shape: (243, 5)
                 Open       High        Low      Close     Volume
Date                                                             
2025-04-21  10.444577  10.568013  10.416092  10.463567  111418418
2025-04-22  10.463567  10.501547  10.425587  10.482557   83112887
2025-04-23  10.482557  10.492052  10.416092  10.454072   58638293
2025-04-24  10.444577  10.501547  10.435082  10.473062   68925613
2025-04-25  10.482557  10.492052  10.435082  10.454072   63754686


## 5. Data Cleaning & Validation

Check for missing values, data types, and any anomalies.  
Baostock data is generally clean, but we explicitly verify.

In [5]:
# Check for missing values
print("Missing values per column:")
print(data.isnull().sum())

# Check data types
print("\nData types:")
print(data.dtypes)

# Check for any zero or negative prices (invalid)
invalid_prices = (data[['Open', 'High', 'Low', 'Close']] <= 0).any().any()
print(f"\nAny invalid (<=0) prices? {invalid_prices}")

# Check for negative volume
negative_volume = (data['Volume'] < 0).any()
print(f"Any negative volume? {negative_volume}")

# If any issues, we would handle them here. In this dataset, none are found.

Missing values per column:
Open      0
High      0
Low       0
Close     0
Volume    0
dtype: int64

Data types:
Open      float64
High      float64
Low       float64
Close     float64
Volume      int64
dtype: object

Any invalid (<=0) prices? False
Any negative volume? False


## 6. Data Transformation

Create derived columns needed for analysis:
- Daily returns
- Cumulative returns
- Rolling moving averages (20-day, 50-day)
- Identify abnormal volume days (2 standard deviations above mean)

In [6]:
# Make a copy to avoid modifying original
df = data.copy()

# Daily return
df['Daily_Return'] = df['Close'].pct_change()

# Cumulative return
df['Cumulative_Return'] = (1 + df['Daily_Return']).cumprod() - 1

# 20-day and 50-day moving averages (if enough data)
if len(df) >= 20:
    df['MA20'] = df['Close'].rolling(window=20).mean()
if len(df) >= 50:
    df['MA50'] = df['Close'].rolling(window=50).mean()

# Abnormal volume: > mean + 2*std
vol_mean = df['Volume'].mean()
vol_std = df['Volume'].std()
df['Is_Abnormal_Volume'] = df['Volume'] > (vol_mean + 2 * vol_std)

print("Transformation complete. New columns added:")
print(df.columns.tolist())

Transformation complete. New columns added:
['Open', 'High', 'Low', 'Close', 'Volume', 'Daily_Return', 'Cumulative_Return', 'MA20', 'MA50', 'Is_Abnormal_Volume']


## 7. Analysis – Risk & Return Metrics

Compute key metrics:
- Annualized volatility
- Maximum drawdown (with peak and trough dates)
- Sharpe ratio (assuming 2% risk‑free rate)
- Total return over the period
- Count of abnormal volume days

In [7]:
def compute_metrics(df):
    """Calculate risk/return metrics from the transformed DataFrame."""
    if df.empty:
        return {}
    
    # Annualized volatility
    daily_vol = df['Daily_Return'].std()
    annual_vol = daily_vol * np.sqrt(252)
    
    # Maximum drawdown
    cumulative = (1 + df['Daily_Return']).cumprod()
    running_max = cumulative.expanding().max()
    drawdown = (cumulative - running_max) / running_max
    max_drawdown = drawdown.min()
    if max_drawdown < 0:
        end_idx = drawdown.idxmin()
        start_idx = cumulative[:end_idx].idxmax()
        max_dd_start = start_idx.strftime('%Y-%m-%d')
        max_dd_end = end_idx.strftime('%Y-%m-%d')
    else:
        max_dd_start = max_dd_end = None
    
    # Sharpe ratio (risk-free rate 2% annual)
    risk_free_rate = 0.02
    excess_return = df['Daily_Return'].mean() * 252 - risk_free_rate
    sharpe_ratio = excess_return / annual_vol if annual_vol != 0 else np.nan
    
    # Total return
    total_return = df['Cumulative_Return'].iloc[-1] if not df.empty else 0
    
    # Abnormal volume days list
    high_volume_days = df[df['Is_Abnormal_Volume']].index.tolist()
    
    metrics = {
        'annual_volatility': annual_vol,
        'max_drawdown': max_drawdown,
        'max_dd_start': max_dd_start,
        'max_dd_end': max_dd_end,
        'sharpe_ratio': sharpe_ratio,
        'total_return': total_return,
        'high_volume_days': high_volume_days,
        'start_date': df.index[0],
        'end_date': df.index[-1]
    }
    return metrics

metrics = compute_metrics(df)
print("Metrics calculated.")

Metrics calculated.


## 8. Output – Key Results with Interpretation

Print the metrics and explain their meaning for the target user.

In [8]:
print("=" * 50)
print(f"Stock: {symbol} ({stock_name})")
print(f"Period: {metrics['start_date'].strftime('%Y-%m-%d')} to {metrics['end_date'].strftime('%Y-%m-%d')}")
print("=" * 50)
print(f"Total Return: {metrics['total_return']:.2%}")
print(f"Annualized Volatility: {metrics['annual_volatility']:.2%}")
print(f"Maximum Drawdown: {metrics['max_drawdown']:.2%} (from {metrics['max_dd_start']} to {metrics['max_dd_end']})")
print(f"Sharpe Ratio: {metrics['sharpe_ratio']:.2f}")
print(f"Abnormal Volume Days: {len(metrics['high_volume_days'])}")
print("=" * 50)

# Interpretation
print("\nInterpretation for retail investors:")
if metrics['total_return'] > 0:
    print("✓ Positive total return – the stock gained value over the period.")
else:
    print("✗ Negative total return – the stock lost value.")
    
if metrics['annual_volatility'] < 0.15:
    print("✓ Low volatility – relatively stable price movements.")
elif metrics['annual_volatility'] < 0.30:
    print("⚠ Moderate volatility – expect average price swings.")
else:
    print("⚠ High volatility – risky, large price fluctuations.")

if metrics['max_drawdown'] < -0.2:
    print(f"⚠ Severe drawdown ({metrics['max_drawdown']:.2%}) – investor could lose >20% at worst.")
    
if not np.isnan(metrics['sharpe_ratio']):
    if metrics['sharpe_ratio'] > 1:
        print("✓ Excellent Sharpe ratio – good risk-adjusted return.")
    elif metrics['sharpe_ratio'] > 0:
        print("✓ Positive Sharpe ratio – return exceeds risk-free rate.")
    else:
        print("✗ Negative Sharpe ratio – underperforms risk-free investment.")

Stock: 000001 (000001)
Period: 2025-04-21 to 2026-04-21
Total Return: 5.89%
Annualized Volatility: 14.85%
Maximum Drawdown: -19.03% (from 2025-07-10 to 2026-03-23)
Sharpe Ratio: 0.34
Abnormal Volume Days: 13

Interpretation for retail investors:
✓ Positive total return – the stock gained value over the period.
✓ Low volatility – relatively stable price movements.
✓ Positive Sharpe ratio – return exceeds risk-free rate.


## 9. Visual Outputs (Plots)

Generate key charts that help users understand the stock’s behaviour.

In [9]:
# Price trend with 20-day MA (if available)
fig1 = go.Figure()
fig1.add_trace(go.Scatter(x=df.index, y=df['Close'], mode='lines', name='Close Price', line=dict(color='blue')))
if 'MA20' in df.columns:
    fig1.add_trace(go.Scatter(x=df.index, y=df['MA20'], mode='lines', name='20-day MA', line=dict(color='orange', dash='dash')))
fig1.update_layout(title=f"{symbol} Price Trend", xaxis_title="Date", yaxis_title="Price (CNY)")
fig1.show()

In [10]:
# Daily return distribution
returns = df['Daily_Return'].dropna()
fig2 = px.histogram(returns, nbins=50, title="Daily Return Distribution", histnorm='probability density')
fig2.add_vline(x=0, line_dash="dash", line_color="red")
fig2.update_layout(xaxis_title="Daily Return", yaxis_title="Probability Density")
fig2.show()

In [11]:
# Cumulative return
fig3 = go.Figure()
fig3.add_trace(go.Scatter(x=df.index, y=df['Cumulative_Return']*100, mode='lines', fill='tozeroy', name='Cumulative Return %'))
fig3.update_layout(title="Cumulative Return (%)", xaxis_title="Date", yaxis_title="Return (%)")
fig3.show()

In [12]:
# Volume with abnormal days highlighted
fig4 = go.Figure()
fig4.add_trace(go.Bar(x=df.index, y=df['Volume'], name='Volume', marker_color='lightblue'))
if len(metrics['high_volume_days']) > 0:
    high_vol_df = df.loc[metrics['high_volume_days']]
    fig4.add_trace(go.Scatter(x=high_vol_df.index, y=high_vol_df['Volume'], mode='markers', name='Abnormal High Volume', marker=dict(color='red', size=8)))
fig4.update_layout(title="Daily Trading Volume", xaxis_title="Date", yaxis_title="Volume (Shares)")
fig4.show()

## 10. Connection to Interactive Tool (Track 4)

All the functions and logic above are directly used in the Streamlit app `app.py`.  
- The notebook serves as **documentation and reproducible core** of the analysis.  
- The app adds **user interaction** (stock code input, date range picker, moving average toggles) and presents the same metrics and charts dynamically.  
- This meets the Track 4 requirement: *“Interactive tool adds real user value”* while the Python analysis remains substantive.

## 11. Limitations & Future Improvements

- **Data limitations:** baostock does not provide company names, dividends, or corporate actions beyond adjusted prices.  
- **Risk‑free rate:** Assumed constant 2% – using a real‑time rate (e.g., 10‑year government bond) would improve Sharpe ratio accuracy.  
- **Single stock only:** Current analysis is per stock; a future version could compare multiple stocks or include sector benchmarks.  
- **No fundamental data:** Only price/volume; adding P/E, P/B, or financial ratios would enrich insights.  
- **Export feature:** Could add PDF/CSV export for users to save reports.